https://gemini.google.com/app/053e07830616e3cd?utm_source=app_launcher&utm_=owned&utm_campaign=base_all

为 Claude Code 定制的生信专属 Skills 矩阵

在 Claude Code 中配置 Skills（通过自定义 Tool 或 MCP Server），核心目的是将生信领域的“脏活累活”（如读取二进制大文件、查询庞大数据库）封装成安全的接口，只把最精简的“摘要结果”返回给大模型。

针对您的 RNA-seq, scRNA-seq 和 WGS 工作流，建议为 Claude Code 装备以下四大核心 Skills：

1. 组学大文件“安全嗅探器” (Data Profiler Skill)

痛点：Claude 无法读取 .bam, .vcf.gz 等二进制文件。如果它试图用 cat 或直接读取大文件，不仅会报错，还会消耗巨量 Token。
Skill 设计：教 Claude 调用专门的探针脚本，而不是直接读取文件。

配置方式：编写一个名为 inspect_genomic_file.sh 的脚本，并在 Claude Code 中注册。

功能：当 Claude 需要了解一个 90GB 的 BAM 文件时，它会调用这个 Skill。该 Skill 自动在底层运行 samtools flagstat 或 bcftools view -H | head -n 10，然后只把几十行的统计摘要返回给 Claude。

效果：Claude 瞬间知道比对率是多少、VCF 包含哪些样本，而消耗的 Token 仅为几分钱。

2. 单细胞/R 对象“透视仪” (Seurat/AnnData Inspector)

痛点：在 Positron 中，您有大量的 .rds 或 .h5ad 单细胞模型。Claude 只能看 .R 脚本，看不到已经跑出来的 R 对象里面到底有哪些细胞亚群（Clusters）。
Skill 设计：为 Claude 提供一个 R/Python 桥梁。

配置方式：编写一个 Python 脚本（利用 anndata 读取）或 R 脚本（利用 Seurat 读取 metadata），封装为 get_sc_metadata Skill。

功能：当您问 Claude “我的 Seurat 对象里有没有免疫细胞？”时，Claude 会自动调用这个 Skill 提取 table(seurat_obj$seurat_clusters) 或提取基因表达矩阵的前 5 行，并返回给大模型。

效果：让终端里的 Claude 仿佛拥有了 Positron 的“Environment（环境变量）”查看面板。

3. 服务器资源“预警机” (Resource Monitor Skill)

痛点：WGS 分析极其吃内存和 I/O。如果您让 Claude 自动跑管线，它如果不看服务器状态直接启动并发，可能会导致 128G 内存爆满，整个服务器宕机。
Skill 设计：赋予 Claude 监控 Linux 硬件的能力。

配置方式：创建一个名为 check_system_health 的 Skill。

功能：在执行重型计算（如 GATK 或 STAR）前，Claude 必须先调用此 Skill 运行 free -h 和 df -h。如果发现可用内存低于 20GB，或者您那 250MB/s 写入的 SSD 空间快满了，Skill 会强制阻止下一步，并让 Claude 提醒您。

效果：让 AI 变成一个懂运维的架构师，保护您宝贵的服务器免受 OOM（Out of Memory）灾难。

4. 基因组数据库“直连探针” (Bio-Database Fetcher)

痛点：在分析 RNA-seq 差异基因时，您经常需要知道某个 Ensembl ID 对应什么基因，或者某个基因在什么通路里。如果让 Claude 直接猜，它可能会产生幻觉。
Skill 设计：将外部公共数据库集成到终端。

配置方式：集成 Entrez Direct (esearch / efetch) 或 MyGene.info API 作为自定义 Skill fetch_gene_info。

功能：当列表中出现未知基因（如 ENSG00000139618）时，Claude 会自动触发 API 请求，从 NCBI 或 Ensembl 获取最新的基因全名和功能注释，然后整合到最终回复中。

效果：无需离开 Positron 终端去查网页，AI 直接帮您完成准确的生物学背景注释。

落地建议

在这些技能中，对您目前解决“AI 工具不能干活又乱花钱”最有帮助的是 前两个（大文件嗅探和 R 对象透视）。它们在 .claudeignore 的物理隔离之外，给 AI 开了一扇安全的小窗户。

https://claude.ai/chat/bdb9929c-9287-4335-a253-7ddd92b810e8

给我输出表格时候, 最终输出都应该包含 | ***|***|***|  这种markown 分割符合。 这样我直接copy 到  ipynb 的markdown 中直接就开业显示良好的格式。 我的这个要求请永远记住并写入setting

这个领域已经发展得相当成熟，Anthropic 官方和社区都有针对你工作场景（RNAseq、单细胞、WGS）的现成 Skills。我按你的三个工作流分类梳理：

一、Anthropic 官方：Claude for Life Sciences Marketplace
这是最值得优先安装的，官方维护、和 Claude Code 深度集成：
bash/plugin marketplace add anthropics/life-sciences
官方提供的 skills 中与你最相关的： 


配套 MCP（不是 skill，但同一市场，建议一起装）：10x-genomics 用于访问 10x Genomics Cloud 分析数据和工作流；pubmed 用于检索生物医学文献。 

二、社区维护：更全面的基因组学 Skill 集合
GoekeLab/awesome-genomic-skills ——一个精选的基因组学和生物信息学 agentic skills、MCP 和 benchmark 列表，覆盖 Claude Code、Copilot、Codex、Cursor、Gemini CLI 等，这是目前最权威的索引仓库，建议先去逛一圈找最新更新。 Positron
GPTomics/bioSkills —— 颗粒度非常细，与你工作直接相关的模块：

CNV 分析：基于读深度的 CNV 检测（CNVkit/GATK）、CBS/HMM 分段、等位基因特异性拷贝数、purity/ploidy 估计、ACMG/ClinGen 胚系 CNV 解读 — WGS 下游分析很实用 GitHub
单倍型分型和基因型 imputation：参考 panel 选择、strand/build 协调、统计学单倍型 phasing、基因型 imputation — WGS 群体遗传分析 GitHub
剪接分析：定量、差异剪接、isoform switching/DTU、sashimi 可视化、单细胞剪接 — RNAseq 进阶分析 GitHub
端到端流程覆盖：RNA-seq、变异检测、ChIP-seq、scRNA-seq、空间转录组、Hi-C 等 GitHub


三、单点工具 Skill（按工具栈选择性安装）
如果你主力用 Python 生态（Scanpy）：

这个 skill 让 Claude 具备处理和分析单细胞 RNA-seq 数据的专门知识，基于 Scanpy 和 AnnData 结构，支持加载 10X、.h5ad 等多种数据格式，执行严格质控、UMAP/t-SNE 降维、识别细胞类型特异性标记基因，还包括轨迹推断和伪时间分析（PAGA、DPT）。 Positron
如果你主力用 R 生态（Seurat/Bioconductor）：

这个 skill 简化 R 中基于 Bioconductor 的复杂生物信息学流程，覆盖 RNA-seq、芯片、单细胞基因组分析，支持用 DESeq2/edgeR 做差异表达分析、用 clusterProfiler 做通路富集，并用 Seurat 处理单细胞数据集、用 ComBat 做批次效应校正。 GitHub

四、自建 Skill（最推荐的长期方案）
如果你想要真正贴合自己实验室惯例（比如固定的 QC 阈值、固定的参考基因组版本），自己写最划算——不需要任何基础设施：
Skills 是 markdown 文件，按需注入领域知识。一套典型的基因组学 skill 集合结构： GitHub
.claude/
├── settings.json
└── skills/
    ├── genomics.md          # 主路由 skill
    ├── genomics-wgs.md       # WGS/WES 指导（含 BWA-MEM2、GATK 流程、Ti/Tv 阈值等）
    ├── genomics-rnaseq.md    # bulk + scRNA-seq
    ├── genomics-qc.md
    └── genomics-cnv.md       # CNV/SV 分析
Skills 能消除生物信息学家的重复性领域上下文输入——不用每个项目都重新跟 Claude 解释参考基因组、样本元数据、QC 阈值或流程参数。一次性把你实验室的最佳实践（比如固定的 QC 阈值、marker 基因发现方法、批次校正参数）固化进可复用的 skill，跨项目通用。这正适合你"日常重复做 RNAseq/scRNAseq/WGS"的场景——把你团队的标准操作流程写成 skill，以后每次分析不用重新交代背景。 Apiyi.com Blog

建议的落地顺序

先装官方 marketplace（single-cell-rna-qc + nextflow-development），立即覆盖你三大场景中的两个
根据你主力用 R 还是 Python，加装对应单点 skill
把你实验室特有的 QC 阈值、参考基因组版本、变异过滤标准写成 1-2 个自建 skill，这是长期收益最高的部分